<a href="https://colab.research.google.com/github/Miguelroots/1.2-Tarea---Clusters-con-SPARK/blob/main/DM_V3_Trabajo_Practico_Regresores_MLlib_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase DM V3 — Trabajo práctico: Aplicaciones de regresores en MLlib con PySpark

**Tema:** Comparación aplicada de modelos de regresión en Apache Spark MLlib.  
**Contexto:** inventario, demanda y logística.  
**Producto esperado:** notebook ejecutado, respuestas argumentadas y conclusión comparativa.

## Propósito de aprendizaje
Al finalizar este trabajo práctico, el estudiante deberá ser capaz de:

1. Preparar un conjunto de datos para regresión usando PySpark.
2. Construir un pipeline de Machine Learning con variables numéricas y categóricas.
3. Aplicar y comparar regresores de MLlib.
4. Interpretar métricas como RMSE, MAE y R².
5. Justificar qué modelo conviene usar según el problema, los datos y la interpretabilidad.

## Instrucciones para el estudiante

Complete las celdas marcadas como **Pregunta para responder** y **Actividad práctica**.  
Puede escribir sus respuestas en celdas Markdown debajo de cada pregunta.

**Regresores que se compararán:**

- Regresión Lineal
- Árbol de Decisión para Regresión
- Random Forest Regressor
- Gradient-Boosted Tree Regressor

**Escenario:** una empresa necesita predecir la **demanda mensual estimada** de productos para mejorar decisiones de almacenamiento, compras y distribución.

In [ ]:
# =========================================================
# 0. Instalación opcional de PySpark
# =========================================================
# En Google Colab, descomente la siguiente línea si PySpark no está instalado:
# !pip install pyspark -q

In [ ]:
# =========================================================
# 1. Crear sesión Spark
# =========================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DM_V3_Regresores_MLlib") \
    .getOrCreate()

spark

# Parte A — Construcción del dataset de trabajo

Para que el ejercicio sea reproducible, se generará un dataset sintético con variables típicas de inventario y demanda.

La variable objetivo será:

**demanda_mensual**: cantidad estimada de unidades demandadas en un mes.

Variables explicativas:

- `categoria_producto`
- `tipo_producto`
- `almacen`
- `precio_unitario`
- `stock_actual`
- `frecuencia_pedidos`
- `dias_entrega_promedio`
- `devoluciones_mes`
- `promocion`
- `estacionalidad`

In [ ]:
# =========================================================
# 2. Generación de datos sintéticos
# =========================================================
import random
import pandas as pd
import numpy as np

random.seed(42)
np.random.seed(42)

n = 1200
categorias = ["Alimentos", "Limpieza", "Tecnologia", "Ropa", "Salud"]
tipos = ["Alta_rotacion", "Media_rotacion", "Baja_rotacion"]
almacenes = ["Norte", "Sur", "Centro", "Costa"]

rows = []

for i in range(n):
    categoria = random.choice(categorias)
    tipo = random.choice(tipos)
    almacen = random.choice(almacenes)
    precio = round(np.random.uniform(3, 250), 2)
    stock = int(np.random.randint(20, 1000))
    frecuencia = int(np.random.randint(1, 30))
    dias_entrega = round(np.random.uniform(1, 15), 1)
    devoluciones = int(np.random.poisson(2))
    promocion = random.choice([0, 1])
    estacionalidad = random.choice([0, 1])

    # Efectos aproximados para simular demanda
    efecto_tipo = {"Alta_rotacion": 130, "Media_rotacion": 70, "Baja_rotacion": 25}[tipo]
    efecto_categoria = {"Alimentos": 80, "Limpieza": 50, "Tecnologia": 35, "Ropa": 45, "Salud": 60}[categoria]
    efecto_almacen = {"Norte": 20, "Sur": 10, "Centro": 35, "Costa": 25}[almacen]

    demanda = (
        30
        + efecto_tipo
        + efecto_categoria
        + efecto_almacen
        + frecuencia * 8
        + promocion * 45
        + estacionalidad * 30
        - precio * 0.18
        - dias_entrega * 2.5
        - devoluciones * 4
        + np.random.normal(0, 25)
    )

    demanda = max(5, round(demanda, 2))

    rows.append([
        i + 1, categoria, tipo, almacen, precio, stock, frecuencia,
        dias_entrega, devoluciones, promocion, estacionalidad, demanda
    ])

columns = [
    "producto_id", "categoria_producto", "tipo_producto", "almacen",
    "precio_unitario", "stock_actual", "frecuencia_pedidos",
    "dias_entrega_promedio", "devoluciones_mes", "promocion",
    "estacionalidad", "demanda_mensual"
]

pdf = pd.DataFrame(rows, columns=columns)
df = spark.createDataFrame(pdf)

df.show(5)

In [ ]:
# Revisar esquema del DataFrame

df.printSchema()

In [ ]:
# Estadísticas descriptivas de variables numéricas

df.select(
    "precio_unitario", "stock_actual", "frecuencia_pedidos",
    "dias_entrega_promedio", "devoluciones_mes", "demanda_mensual"
).describe().show()

## Pregunta 1 — Comprensión del problema

Explique con sus palabras:

1. ¿Qué representa la variable objetivo `demanda_mensual`?
2. ¿Por qué este problema puede tratarse como un problema de regresión y no como clasificación?
3. ¿Qué decisión empresarial podría mejorar si el modelo predice bien la demanda?

**Respuesta del estudiante:**

> Escriba aquí su respuesta.

# Parte B — Preparación de datos para MLlib

En Spark MLlib, los modelos no reciben columnas separadas como entrada. Primero se debe construir una columna vectorial llamada normalmente `features`.

Además, las variables categóricas deben transformarse a representaciones numéricas usando:

- `StringIndexer`
- `OneHotEncoder`
- `VectorAssembler`

In [ ]:
# =========================================================
# 3. División entrenamiento / prueba
# =========================================================
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("Registros de entrenamiento:", train_df.count())
print("Registros de prueba:", test_df.count())

In [ ]:
# =========================================================
# 4. Preparación del pipeline de variables
# =========================================================
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

categorical_cols = ["categoria_producto", "tipo_producto", "almacen"]
numeric_cols = [
    "precio_unitario", "stock_actual", "frecuencia_pedidos",
    "dias_entrega_promedio", "devoluciones_mes", "promocion", "estacionalidad"
]

indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_idx", handleInvalid="keep")
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=col + "_idx", outputCol=col + "_ohe")
    for col in categorical_cols
]

assembler_inputs = numeric_cols + [col + "_ohe" for col in categorical_cols]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

## Pregunta 2 — Preparación de variables

Responda:

1. ¿Por qué no se puede usar directamente una variable como `categoria_producto` en un modelo de MLlib?
2. ¿Qué hace `StringIndexer`?
3. ¿Qué hace `OneHotEncoder`?
4. ¿Por qué puede ser útil escalar variables numéricas?

**Respuesta del estudiante:**

> Escriba aquí su respuesta.

# Parte C — Aplicación 1: Regresión Lineal

La regresión lineal es útil cuando se espera una relación aproximadamente lineal entre las variables explicativas y la variable objetivo.

Ventajas:

- Fácil de interpretar.
- Rápida de entrenar.
- Sirve como modelo base.

Limitaciones:

- Puede funcionar mal si la relación entre variables es no lineal.
- Es sensible a valores atípicos y correlaciones fuertes entre variables.

In [ ]:
# =========================================================
# 5. Modelo 1: Regresión Lineal
# =========================================================
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(
    featuresCol="features",
    labelCol="demanda_mensual",
    predictionCol="prediccion_lr"
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])
model_lr = pipeline_lr.fit(train_df)
pred_lr = model_lr.transform(test_df)

pred_lr.select("demanda_mensual", "prediccion_lr").show(10)

In [ ]:
# =========================================================
# 6. Evaluación de Regresión Lineal
# =========================================================
def evaluar(predictions, prediction_col):
    evaluator_rmse = RegressionEvaluator(
        labelCol="demanda_mensual", predictionCol=prediction_col, metricName="rmse"
    )
    evaluator_mae = RegressionEvaluator(
        labelCol="demanda_mensual", predictionCol=prediction_col, metricName="mae"
    )
    evaluator_r2 = RegressionEvaluator(
        labelCol="demanda_mensual", predictionCol=prediction_col, metricName="r2"
    )
    return {
        "RMSE": evaluator_rmse.evaluate(predictions),
        "MAE": evaluator_mae.evaluate(predictions),
        "R2": evaluator_r2.evaluate(predictions)
    }

metricas_lr = evaluar(pred_lr, "prediccion_lr")
metricas_lr

## Pregunta 3 — Interpretación de la regresión lineal

Responda:

1. ¿El valor de R² indica que el modelo explica bien la demanda? Justifique.
2. ¿Qué significa un RMSE alto en este contexto?
3. ¿En qué caso empresarial preferiría usar regresión lineal aunque otro modelo tenga mejor precisión?

**Respuesta del estudiante:**

> Escriba aquí su respuesta.

# Parte D — Aplicación 2: Árbol de Decisión para Regresión

Un árbol de decisión divide los datos en reglas. Por ejemplo, puede aprender que los productos de alta rotación con promoción tienen mayor demanda.

Ventajas:

- Captura relaciones no lineales.
- Es más interpretable que modelos ensamblados.
- No requiere tanta preparación estadística como una regresión lineal.

Limitaciones:

- Puede sobreajustarse.
- Pequeños cambios en los datos pueden generar árboles diferentes.

In [ ]:
# =========================================================
# 7. Modelo 2: Decision Tree Regressor
# =========================================================
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="demanda_mensual",
    predictionCol="prediccion_dt",
    maxDepth=5,
    seed=42
)

pipeline_dt = Pipeline(stages=indexers + encoders + [assembler, scaler, dt])
model_dt = pipeline_dt.fit(train_df)
pred_dt = model_dt.transform(test_df)

pred_dt.select("demanda_mensual", "prediccion_dt").show(10)

In [ ]:
metricas_dt = evaluar(pred_dt, "prediccion_dt")
metricas_dt

## Pregunta 4 — Árbol de decisión

Responda:

1. ¿Por qué un árbol puede capturar relaciones que una regresión lineal no captura fácilmente?
2. ¿Qué riesgo aparece si se aumenta demasiado `maxDepth`?
3. Proponga una regla de negocio que un árbol podría aprender en este problema.

**Respuesta del estudiante:**

> Escriba aquí su respuesta.

# Parte E — Aplicación 3: Random Forest Regressor

Random Forest combina muchos árboles de decisión. Cada árbol aprende con una muestra diferente de los datos y luego se promedian sus predicciones.

Ventajas:

- Suele mejorar la generalización.
- Reduce el sobreajuste de un solo árbol.
- Funciona bien con relaciones no lineales.

Limitaciones:

- Menos interpretable que un árbol individual.
- Puede requerir más recursos computacionales.

In [ ]:
# =========================================================
# 8. Modelo 3: Random Forest Regressor
# =========================================================
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="demanda_mensual",
    predictionCol="prediccion_rf",
    numTrees=50,
    maxDepth=6,
    seed=42
)

pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])
model_rf = pipeline_rf.fit(train_df)
pred_rf = model_rf.transform(test_df)

pred_rf.select("demanda_mensual", "prediccion_rf").show(10)

In [ ]:
metricas_rf = evaluar(pred_rf, "prediccion_rf")
metricas_rf

## Pregunta 5 — Random Forest

Responda:

1. ¿Por qué Random Forest puede ser más estable que un solo árbol de decisión?
2. ¿Qué impacto tendría aumentar `numTrees`?
3. ¿Por qué este modelo podría ser útil en datos masivos?

**Respuesta del estudiante:**

> Escriba aquí su respuesta.

# Parte F — Aplicación 4: Gradient-Boosted Tree Regressor

Gradient-Boosted Trees construye árboles de forma secuencial. Cada nuevo árbol intenta corregir los errores de los anteriores.

Ventajas:

- Suele lograr alta precisión.
- Captura relaciones complejas.
- Es competitivo para muchos problemas tabulares.

Limitaciones:

- Puede ser más lento.
- Requiere cuidado con hiperparámetros.
- Puede sobreajustarse si se configura mal.

In [ ]:
# =========================================================
# 9. Modelo 4: Gradient-Boosted Tree Regressor
# =========================================================
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="demanda_mensual",
    predictionCol="prediccion_gbt",
    maxIter=40,
    maxDepth=4,
    seed=42
)

pipeline_gbt = Pipeline(stages=indexers + encoders + [assembler, scaler, gbt])
model_gbt = pipeline_gbt.fit(train_df)
pred_gbt = model_gbt.transform(test_df)

pred_gbt.select("demanda_mensual", "prediccion_gbt").show(10)

In [ ]:
metricas_gbt = evaluar(pred_gbt, "prediccion_gbt")
metricas_gbt

## Pregunta 6 — Gradient-Boosted Trees

Responda:

1. ¿Qué diferencia principal existe entre Random Forest y Gradient-Boosted Trees?
2. ¿Por qué GBT puede tener mejor precisión que otros modelos?
3. ¿Qué problema puede aparecer si el modelo se entrena con demasiadas iteraciones?

**Respuesta del estudiante:**

> Escriba aquí su respuesta.

# Parte G — Comparación general de regresores

Ahora se consolidan las métricas para comparar los modelos.

In [ ]:
# =========================================================
# 10. Comparación de métricas
# =========================================================
resultados = pd.DataFrame([
    {"Modelo": "Regresión Lineal", **metricas_lr},
    {"Modelo": "Árbol de Decisión", **metricas_dt},
    {"Modelo": "Random Forest", **metricas_rf},
    {"Modelo": "Gradient-Boosted Trees", **metricas_gbt},
])

resultados

In [ ]:
# =========================================================
# 11. Ordenar modelos por RMSE menor
# =========================================================
resultados.sort_values("RMSE")

In [ ]:
# =========================================================
# 12. Gráfico simple de comparación
# =========================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.bar(resultados["Modelo"], resultados["RMSE"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("RMSE")
plt.title("Comparación de modelos según RMSE")
plt.show()

## Pregunta 7 — Comparación crítica

Complete la tabla con su interpretación:

| Modelo | Ventaja principal | Limitación principal | ¿Lo usaría en producción? ¿Por qué? |
|---|---|---|---|
| Regresión Lineal |  |  |  |
| Árbol de Decisión |  |  |  |
| Random Forest |  |  |  |
| Gradient-Boosted Trees |  |  |  |

**Respuesta del estudiante:**

> Complete la tabla.

# Parte H — Actividades prácticas adicionales

## Actividad práctica 1 — Cambio de variable objetivo

Cambie la variable objetivo y construya un nuevo modelo para predecir una de las siguientes variables:

1. `dias_entrega_promedio`
2. `stock_actual`
3. Una nueva variable creada por usted, por ejemplo `demanda_alta_valor = demanda_mensual * precio_unitario`

Explique si sigue siendo un problema de regresión y qué utilidad tendría para la empresa.

In [ ]:
# Actividad práctica 1
# Cree aquí una nueva variable objetivo o cambie el labelCol del modelo.

# Ejemplo sugerido:
# from pyspark.sql.functions import col
# df2 = df.withColumn("valor_demanda", col("demanda_mensual") * col("precio_unitario"))
# Luego adapte los pipelines anteriores usando labelCol="valor_demanda".

## Actividad práctica 2 — Ajuste de hiperparámetros

Modifique al menos dos hiperparámetros y compare resultados:

- `maxDepth` en Decision Tree
- `numTrees` en Random Forest
- `maxIter` en GBT
- `regParam` en Regresión Lineal

Debe responder:

1. ¿Qué hiperparámetro modificó?
2. ¿Mejoró o empeoró el RMSE?
3. ¿El cambio hizo al modelo más simple o más complejo?
4. ¿Qué configuración recomendaría?

In [ ]:
# Actividad práctica 2
# Copie uno de los modelos anteriores y modifique hiperparámetros.
# Compare los resultados con la tabla original.

## Actividad práctica 3 — Análisis de errores

Seleccione 10 predicciones del mejor modelo y compare el valor real con el valor predicho.

Responda:

1. ¿En qué casos el modelo se equivoca más?
2. ¿Existe algún patrón en los errores?
3. ¿Qué variable adicional podría ayudar a mejorar la predicción?

In [ ]:
# Actividad práctica 3
# Use el modelo que obtuvo mejor RMSE.
# Ejemplo con GBT:

pred_gbt.select(
    "producto_id", "categoria_producto", "tipo_producto", "almacen",
    "demanda_mensual", "prediccion_gbt"
).show(10, truncate=False)

# Parte I — Preguntas finales de reflexión

Responda de forma argumentada:

1. ¿Qué modelo obtuvo mejor desempeño según RMSE?
2. ¿Coincide el mejor RMSE con el modelo más interpretable?
3. Si usted fuera analista de datos de la empresa, ¿qué modelo recomendaría y por qué?
4. ¿Qué riesgos existen al usar un modelo de regresión sin validar los datos de entrada?
5. ¿Cómo se relaciona este ejercicio con el ciclo de vida de un proyecto de ciencia de datos?
6. ¿Qué ventaja ofrece PySpark frente a pandas cuando los datos crecen a millones de registros?

**Respuesta final del estudiante:**

> Escriba aquí su conclusión final.

# Rúbrica sugerida para calificación sobre 10 puntos

| Criterio | Puntaje |
|---|---:|
| Ejecución correcta del notebook y preparación de datos | 2.0 |
| Aplicación de al menos cuatro regresores en MLlib | 2.0 |
| Interpretación correcta de RMSE, MAE y R² | 2.0 |
| Comparación crítica de modelos | 1.5 |
| Actividades prácticas adicionales | 1.5 |
| Conclusión final argumentada | 1.0 |
| **Total** | **10.0** |

# Cierre docente

Este trabajo práctico permite conectar la infografía conceptual con una aplicación real en PySpark. La idea central no es solo ejecutar modelos, sino comprender cuándo conviene usar cada regresor, cómo se evalúa su desempeño y qué implicaciones tiene para una decisión empresarial.